In [ ]:
import os
import re
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

import pandas as pd


# =========================
# パス
# =========================
BASE = Path(os.environ.get("OneDrive", "")) / "タブレット用"
DB_PATH = BASE / "recipe_db.xlsx"
SHEET_NAME = "Recipes"

OUT_XLSX = BASE / "dup_name_and_url_strict.xlsx"


# =========================
# 正規化
# =========================
def norm_name(x: str) -> str:
    """レシピ名の比較用正規化"""
    if x is None:
        return ""
    s = str(x).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

def clean_url(u: str) -> str:
    """URL比較用（? と # を除去）"""
    u = (u or "").strip()
    if not u:
        return ""
    sp = urlsplit(u)
    return urlunsplit((sp.scheme, sp.netloc, sp.path, "", ""))


# =========================
# メイン
# =========================
def main():
    if not DB_PATH.exists():
        raise FileNotFoundError(f"DBが見つかりません: {DB_PATH}")

    df = pd.read_excel(DB_PATH, sheet_name=SHEET_NAME)

    # 必須列
    for c in ["RecipeID", "レシピ名", "URL"]:
        if c not in df.columns:
            raise ValueError(f"{SHEET_NAME} に列 {c} がありません")

    # 正規化キー作成
    df["RecipeID"] = df["RecipeID"].astype(str).str.strip()
    df["name_key"] = df["レシピ名"].map(norm_name)
    df["url_key"] = df["URL"].map(clean_url)

    # URLが空のものは除外（URL一致が条件なので）
    df2 = df[df["url_key"] != ""].copy()

    # ★ 重複判定：name_key + url_key 完全一致
    dup = df2[df2.duplicated(subset=["name_key", "url_key"], keep=False)].copy()

    # 見やすく
    dup = dup.sort_values(["url_key", "name_key", "RecipeID"])

    # 出力
    with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
        df2[["RecipeID", "レシピ名", "URL", "name_key", "url_key"]].to_excel(
            w, sheet_name="DB_KEY", index=False
        )

        if dup.empty:
            pd.DataFrame(
                [{"message": "（レシピ名＋URL）が完全一致する重複はありません"}]
            ).to_excel(w, sheet_name="DUPLICATES", index=False)
        else:
            dup[["RecipeID", "レシピ名", "URL"]].to_excel(
                w, sheet_name="DUPLICATES", index=False
            )

    print(f"Saved: {OUT_XLSX}")
    print(f"Duplicate rows: {len(dup)}")


if __name__ == "__main__":
    main()



AttributeError: 'float' object has no attribute 'strip'